# Drought Index Analysis

Computing SPI and SPEI to quantify drought severity, duration, and frequency across the 45 years in the analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
bcolor = "darkslategrey"

df = pd.read_csv("../data/processed/mendoza_basin_monthly.csv", parse_dates=["time"], index_col="time")

print(f"Loaded {len(df)} months ({df.index[0].strftime('%Y-%m')} to {df.index[-1].strftime('%Y-%m')})")
df.head()

Loaded 540 months (1980-01 to 2024-12)


,precip_mm,temp_c,pev_mm,runoff_mm,soil_moisture_0_7cm
time,,,,,
1980-01-01,63.820649,15.302368,183.912286,98.165597,0.191278
1980-02-01,93.886005,13.928650,148.222182,36.150087,0.191747
1980-03-01,66.787061,14.917786,132.601721,29.331790,0.162673
1980-04-01,162.947316,6.125000,59.435077,35.352784,0.201941
1980-05-01,115.186830,3.713196,39.293950,28.664771,0.218384


## SPI Computation

For the data, I want to calculate the SPI 3, 6, and 12 of the data. To make this simpler, I am going to make an SPI function. The steps for this will be as follows: 

- Rolling sum of precipitation over the starting month (so for SPI3 of August 2020, it would be the sum of June, July, and August 2020)  
- For each calendar month, do this and fit a gamma distribution to all historical values    
- Transform through gammaCDF, inverse normal CDF, and then you have the SPI value   


In [ ]:
def compute_spi(precip_series, window):
    # Calculating the rolling sum of each calendar month
    precip_acc = precip_series.rolling(window=window, min_periods=window).sum()
    spi = pd.Series(index=precip_series.index, dtype=float)

    for month in range(1, 13):
        month_mask = precip_acc.index.month == month
        month_data = precip_acc[month_mask].dropna()

        # This is all to clean up the data, if there is too few data points it doesn't fit the gamma distribution, and if there are too few nonzero values same thing
        # This is also used to then later calculate what the non zero values are with the gamma distribution
        if len(month_data) < 10:
            continue
        nonzero = month_data[month_data > 0]
        prob_zero = 1.0 - len(nonzero) / len(month_data)
        if len(nonzero) < 5:
            continue
        
        # Fitting it nonzeros to a gamma distribution
        try:
            alpha, loc, beta = stats.gamma.fit(nonzero, floc=0)
        except Exception:
            continue

        for idx in month_data.index:
            val = precip_acc[idx]
            if np.isnan(val):
                continue
            
            # This is transforming it through gammaCDF
            if val <= 0:
                cdf_val = prob_zero / 2
            else:
                gamma_cdf = stats.gamma.cdf(val, alpha, loc=0, scale=beta)
                cdf_val = prob_zero + (1 - prob_zero) * gamma_cdf
            cdf_val = np.clip(cdf_val, 0.001, 0.999)

            # Transforms through inverse normal to give the spi value 
            spi[idx] = stats.norm.ppf(cdf_val)

    return spi

spi_3 = compute_spi(df["precip_mm"], window=3)
spi_6 = compute_spi(df["precip_mm"], window=6)
spi_12 = compute_spi(df["precip_mm"], window=12)

print("SPI Valid Computations:")
print(f"SPI3: {spi_3.notna().sum()} valid months")
print(f"SPI6: {spi_6.notna().sum()} valid months")
print(f"SPI12: {spi_12.notna().sum()} valid months")

# Checking that they were all standardised correctly
for name, spi in [("SPI-3", spi_3), ("SPI-6", spi_6), ("SPI-12", spi_12)]:
    valid = spi.dropna()
    print(f"\n{name} mean: {valid.mean():.3f}, std: {valid.std():.3f}")

SPI Valid Computations:
SPI3: 538 valid months
SPI6: 535 valid months
SPI12: 529 valid months

SPI-3 mean: -0.001, std: 1.001

SPI-6 mean: -0.001, std: 1.001

SPI-12 mean: -0.000, std: 1.001
